In [13]:
!pip install -q peft

In [14]:
import os, gc, random, warnings
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.model_selection import StratifiedKFold
from transformers import (
    AutoTokenizer, AutoModelForMultipleChoice,
    TrainingArguments, Trainer, DefaultDataCollator, set_seed
)
from peft import get_peft_model, LoraConfig, TaskType

warnings.filterwarnings('ignore')

class Config:
    MODEL_NAME = 'roberta-large'
    MAX_LEN    = 160
    BATCH_SIZE = 4
    GRAD_ACCUM = 4
    EPOCHS     = 3
    LR         = 3e-5
    N_FOLDS    = 5
    SEED       = 42
    USE_TTA    = True
    TTA_PREFIX = 'Analyze and solve this multiple-choice question carefully: '

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    set_seed(seed)

seed_everything(Config.SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device} | Model: {Config.MODEL_NAME} | torch: {torch.__version__}')

Device: cuda | Model: roberta-large | torch: 2.10.0+cu128


In [15]:
def find_path(filename):
    for p in [
        f'/kaggle/input/competitions/smart-mcq-solver-challenge/{filename}',
        f'/kaggle/input/smart-mcq-solver-challenge/{filename}',
        filename
    ]:
        if os.path.exists(p):
            return p
    return None

train_df = pd.read_csv(find_path('train.csv'))
test_path = find_path('test.csv')
test_df  = pd.read_csv(test_path) if test_path else train_df.copy()

for col in ['prompt','A','B','C','D','E']:
    train_df[col] = train_df[col].fillna('').astype(str)
    test_df[col]  = test_df[col].fillna('').astype(str)

label_map     = {'A':0,'B':1,'C':2,'D':3,'E':4}
inv_label_map = {v:k for k,v in label_map.items()}
train_df['label'] = train_df['answer'].map(label_map)
print(f'Train: {len(train_df)} | Test: {len(test_df)}')

Train: 2000 | Test: 500


In [16]:
tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME)
print(f"Pad token: '{tokenizer.pad_token}' (ID: {tokenizer.pad_token_id})")
assert tokenizer.pad_token_id is not None

def preprocess_function(examples, prepend_text=''):
    first  = [[prepend_text + ctx]*5 for ctx in examples['prompt']]
    second = [[examples[opt][i] for opt in ['A','B','C','D','E']]
              for i in range(len(examples['prompt']))]
    tok = tokenizer(sum(first,[]), sum(second,[]),
                    truncation=True, max_length=Config.MAX_LEN, padding='max_length')
    return {k:[v[i:i+5] for i in range(0,len(v),5)] for k,v in tok.items()}

def mapk(actuals, predictions, k=3):
    scores = []
    for a, p in zip(actuals, predictions):
        score = 0.0
        for rank, pred in enumerate(p[:k], 1):
            if pred == a:
                score = 1.0/rank
                break
        scores.append(score)
    return float(np.mean(scores))

Pad token: '<pad>' (ID: 1)


In [17]:
COLS_KEEP = ['input_ids','attention_mask']

def make_test_ds(prepend=''):
    ds = Dataset.from_pandas(test_df).map(
        lambda x: preprocess_function(x, prepend), batched=True)
    return ds.remove_columns(
        [c for c in ds.column_names if c not in COLS_KEEP]
    ).with_format('torch')

test_ds_std = make_test_ds('')
test_ds_tta = make_test_ds(Config.TTA_PREFIX)
print('Test datasets ready:', test_ds_std.column_names)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Test datasets ready: ['input_ids', 'attention_mask']


In [18]:
skf = StratifiedKFold(n_splits=Config.N_FOLDS, shuffle=True, random_state=Config.SEED)
oof_probs        = np.zeros((len(train_df), 5))
test_probs_folds = np.zeros((len(test_df), 5))

for fold, (tr_idx, val_idx) in enumerate(skf.split(train_df, train_df['label'])):
    print(f"\n{'='*20} FOLD {fold+1}/{Config.N_FOLDS} {'='*20}")

    tr_fold  = train_df.iloc[tr_idx].reset_index(drop=True)
    val_fold = train_df.iloc[val_idx].reset_index(drop=True)

    def make_split_ds(df):
        ds = Dataset.from_pandas(df).map(
            lambda x: preprocess_function(x, ''), batched=True)
        ds = ds.rename_column('label', 'labels')
        keep = COLS_KEEP + ['labels']
        return ds.remove_columns(
            [c for c in ds.column_names if c not in keep]
        ).with_format('torch')

    tr_ds  = make_split_ds(tr_fold)
    val_ds = make_split_ds(val_fold)

    base = AutoModelForMultipleChoice.from_pretrained(Config.MODEL_NAME)
    base.config.pad_token_id = tokenizer.pad_token_id

    lora_cfg = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.1,
        target_modules=['query','value','key','dense'],
        bias='none', task_type=TaskType.SEQ_CLS
    )
    model = get_peft_model(base, lora_cfg).to(device)
    model.print_trainable_parameters()

    args = TrainingArguments(
        output_dir=f'./fold_{fold}',
        per_device_train_batch_size=Config.BATCH_SIZE,
        per_device_eval_batch_size=Config.BATCH_SIZE*2,
        gradient_accumulation_steps=Config.GRAD_ACCUM,
        num_train_epochs=Config.EPOCHS,
        learning_rate=Config.LR,
        warmup_steps=50,
        lr_scheduler_type='cosine',
        fp16=torch.cuda.is_available(),
        logging_steps=20,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        report_to='none'
    )

    # NOTE: 'tokenizer' arg removed — not supported in this transformers version
    trainer = Trainer(
        model=model, args=args,
        train_dataset=tr_ds, eval_dataset=val_ds,
        data_collator=DefaultDataCollator()
    )
    trainer.train()

    # OOF evaluation
    val_out   = trainer.predict(val_ds)
    val_probs = torch.softmax(torch.tensor(val_out.predictions), dim=-1).numpy()
    oof_probs[val_idx] = val_probs
    val_top3  = np.argsort(val_probs, axis=1)[:,::-1][:,:3]
    fold_map3 = mapk(val_fold['answer'].tolist(),
                     [[inv_label_map[i] for i in row] for row in val_top3])
    print(f'  Fold {fold+1} OOF MAP@3 = {fold_map3:.4f}')

    # Test inference + TTA
    std_probs = torch.softmax(torch.tensor(
        trainer.predict(test_ds_std).predictions), dim=-1).numpy()
    if Config.USE_TTA:
        tta_probs = torch.softmax(torch.tensor(
            trainer.predict(test_ds_tta).predictions), dim=-1).numpy()
        fold_test = (std_probs + tta_probs) / 2
    else:
        fold_test = std_probs
    test_probs_folds += fold_test / Config.N_FOLDS

    del model, base, trainer, tr_ds, val_ds
    gc.collect(); torch.cuda.empty_cache()


==================== FOLD 1/5 ====================


Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaForMultipleChoice LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.pooler.dense.weight     | MISSING    | 
roberta.pooler.dense.bias       | MISSING    | 
classifier.weight               | MISSING    | 
classifier.bias                 | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaForMultipleChoice LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.pooler.dense.weight     | MISSING    | 
roberta.pooler.dense.bias       | MISSING    | 
classifier.weight               | MISSING    | 
classifier.bias                 | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 7,111,681 || all params: 362,472,450 || trainable%: 1.9620


Epoch,Training Loss,Validation Loss
1,12.908659,3.190328
2,12.729901,3.144547
3,12.802782,3.135576


  Fold 5 OOF MAP@3 = 0.5450


In [19]:
oof_top3   = np.argsort(oof_probs, axis=1)[:,::-1][:,:3]
oof_labels = [[inv_label_map[i] for i in row] for row in oof_top3]
final_map3 = mapk(train_df['answer'].tolist(), oof_labels)

print('='*55)
print(f'FINAL OOF MAP@3 SCORE: {final_map3:.4f}')
print('='*55)
print('TARGET REACHED!' if final_map3 >= 0.73 else 'Keep tuning!')

test_top3 = np.argsort(test_probs_folds, axis=1)[:,::-1][:,:3]
preds = [' '.join([inv_label_map[i] for i in row]) for row in test_top3]

sub = pd.DataFrame({
    'id': test_df['id'] if 'id' in test_df.columns else range(len(test_df)),
    'prediction': preds
})
sub.to_csv('submission.csv', index=False)
print('submission.csv saved!')
sub.head(10)

FINAL OOF MAP@3 SCORE: 0.5302
Keep tuning!
submission.csv saved!


,id,prediction
0,1,B A D
1,2,E A D
2,3,E B C
3,4,D B E
4,5,D C B
5,6,E A D
6,7,C B A
7,8,A D C
8,9,B D A
9,10,C B E
